# 12 · Profile: CPI-U, New York metro, for pay in today's dollars

**Method:** `docs/framework.md` §3. Card: `docs/source-cards/bls_cpi.md`.

**Why this source matters.** brief question 4 compares pay over time (OEWS May 2018 → May 2025, nb 03; ACS wages, nb 10). A dollar in 2019 bought more than a dollar in 2025, so pay over time has to be shown in the same year's dollars (brief O4: "OEWS medians adjusted with CPI"). The BLS Consumer Price Index for All Urban Consumers (CPI-U) for the New York-Newark-Jersey City area is the standard way to do that for NYC.

**What this notebook does:** reads the BLS CPI flat files, finds the NY series, checks grain, periods, gaps and the annual averages, and builds the deflator the dashboard would use. It changes nothing.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import RAW

pd.set_option("display.max_columns", 40, "display.max_colwidth", 90, "display.width", 220)
man = pd.read_csv(RAW / "manifest.csv", dtype=str, keep_default_na=False)
FILES = man[man.key == "bls_cu"][["key", "file", "downloaded_utc", "bytes", "note"]]
display(FILES)
P = {Path(f).name.split("_")[0]: RAW / f for f in FILES.file}

,key,file,downloaded_utc,bytes,note
138,bls_cu,bls_timeseries/bls_cu/cu-data-1-AllItems_2026-10-12.txt,2026-10-12T17:23:16+00:00,2688415,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
139,bls_cu,bls_timeseries/bls_cu/cu-series_2026-10-12.txt,2026-10-12T17:23:17+00:00,1339447,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
140,bls_cu,bls_timeseries/bls_cu/cu-area_2026-10-12.txt,2026-10-12T17:23:18+00:00,2185,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
141,bls_cu,bls_timeseries/bls_cu/cu-item_2026-10-12.txt,2026-10-12T17:23:19+00:00,16554,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
142,bls_cu,bls_timeseries/bls_cu/cu-period_2026-10-12.txt,2026-10-12T17:23:20+00:00,323,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
143,bls_cu,bls_timeseries/bls_cu/cu-footnote_2026-10-12.txt,2026-10-12T17:23:21+00:00,100,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"
144,bls_cu,bls_timeseries/bls_cu/cu-txt_2026-10-12.txt,2026-10-12T17:23:22+00:00,14644,"CPI-U all items by area (NY-Newark-Jersey City = S12A), for real-dollar pay"


## 0. Source header

| | **BLS Consumer Price Index, CPI-U** (`bls_cu`) |
|---|---|
| **Publisher** | U.S. Bureau of Labor Statistics, Division of Consumer Prices and Price Indexes |
| **Files** | BLS flat files from `download.bls.gov/pub/time.series/cu/`: `cu.data.1.AllItems` (every *all items* series, all areas) + the lookup files (`series`, `area`, `item`, `period`, `footnote`) and the `cu.txt` readme. Saved with the download date in the name (the files overwrite themselves) |
| **Release** | Downloaded 2026-10-12; latest month August 2026 |
| **How it was collected** | **Survey.** Prices of a fixed basket of goods and services, collected monthly in 75 urban areas; NY metro is one of the published areas |
| **Index** | 1982–84 = 100. Not seasonally adjusted for metro areas (`CUUR…`); BLS publishes no seasonally adjusted metro series |
| **License** | Public domain |
| **Documentation** | [CPI handbook of methods](https://www.bls.gov/opub/hom/cpi/), [`cu.txt`](https://download.bls.gov/pub/time.series/cu/cu.txt) |

## 1. Load as text

In [2]:
rd = lambda k: pd.read_csv(P[k], sep="\t", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip()).rename(columns=str.strip)
data, series, area, period, foot = rd("cu-data-1-AllItems"), rd("cu-series"), rd("cu-area"), rd("cu-period"), rd("cu-footnote")
for n, d in {"data": data, "series": series, "area": area, "period": period, "footnote": foot}.items():
    print(f"{n:<9} {d.shape[0]:>7,} rows x {d.shape[1]} cols | {list(d.columns)}")
print("\nNY area:", area[area.area_code == "S12A"].values.tolist())
ny = series[(series.area_code == "S12A") & (series.item_code == "SA0")]
display(ny[["series_id", "seasonal", "periodicity_code", "base_period", "series_title", "begin_year", "begin_period", "end_year", "end_period"]])

data       64,007 rows x 5 cols | ['series_id', 'year', 'period', 'value', 'footnote_codes']
series      8,104 rows x 13 cols | ['series_id', 'area_code', 'item_code', 'seasonal', 'periodicity_code', 'base_code', 'base_period', 'series_title', 'footnote_codes', 'begin_year', 'begin_period', 'end_year', 'end_period']
area           58 rows x 5 cols | ['area_code', 'area_name', 'display_level', 'selectable', 'sort_sequence']
period         16 rows x 3 cols | ['period', 'period_abbr', 'period_name']
footnote        2 rows x 2 cols | ['footnote_code', 'footnote_text']

NY area: [['S12A', 'New York-Newark-Jersey City, NY-NJ-PA', '1', 'T', '6']]


,series_id,seasonal,periodicity_code,base_period,series_title,begin_year,begin_period,end_year,end_period
2655,CUURS12ASA0,U,R,1982-84=100,"All items in New York-Newark-Jersey City, NY-NJ-PA, all urban consumers, not seasonall...",1914,M12,2026,M08
6556,CUUSS12ASA0,U,S,1982-84=100,"All items in New York-Newark-Jersey City, NY-NJ-PA, all urban consumers, not seasonall...",1984,S01,2026,S01


**Load findings**
- The all-items file has 201 series for 58 areas, 64,007 rows. New York-Newark-Jersey City is area `S12A` (it covers NYC plus suburbs in NY, NJ and PA; there is no NYC-only CPI).
- Two NY all-items series: `CUURS12ASA0` monthly (from 1914, latest August 2026) and `CUUSS12ASA0` half-yearly. Both not seasonally adjusted, 1982–84 = 100.

## 2. What one row is (grain)

**Presumed key:** `series_id + year + period`.

In [3]:
print("Duplicates on series_id + year + period:", data.duplicated(["series_id", "year", "period"]).sum())
print(f"{data.series_id.nunique()} series in the all-items file; areas: {data.series_id.str[4:8].nunique()}")
cpi = data[data.series_id.isin(["CUURS12ASA0", "CUUSS12ASA0", "CUUR0000SA0"])].copy()
cpi["year"] = cpi.year.astype(int)
cpi["value_num"] = pd.to_numeric(cpi.value, errors="coerce")
display(cpi.groupby("series_id").agg(rows=("value", "size"), first=("year", "min"), last=("year", "max"),
                                    periods=("period", lambda s: ",".join(sorted(s.unique())[:3]) + " … " + ",".join(sorted(s.unique())[-2:]))))

Duplicates on series_id + year + period: 0
201 series in the all-items file; areas: 58


,rows,first,last,periods
series_id,,,,
CUUR0000SA0,1477,1913,2026,"M01,M02,M03 … M12,M13"
CUURS12ASA0,1209,1914,2026,"M01,M02,M03 … M12,M13"
CUUSS12ASA0,127,1984,2026,"S01,S02,S03 … S02,S03"


**Grain findings**
- Key `series_id + year + period` is unique . One row = one series × one period. Periods `M01–M12` are months, `M13` is the annual average; `S01/S02` halves, `S03` annual average.

## 3–4. Profile, periods and special values

In [4]:
display(period)
display(foot)
bad = cpi[cpi.value_num.isna()]
print("Non-numeric values:"); display(bad.merge(foot, left_on="footnote_codes", right_on="footnote_code", how="left")
                                     [["series_id", "year", "period", "value", "footnote_codes", "footnote_text"]])
print("Footnote codes used in the three series:", cpi.footnote_codes.value_counts().to_dict())

,period,period_abbr,period_name
0,M01,JAN,January
1,M02,FEB,February
2,M03,MAR,March
3,M04,APR,April
4,M05,MAY,May
5,M06,JUN,June
6,M07,JUL,July
7,M08,AUG,August
8,M09,SEP,September
9,M10,OCT,October


,footnote_code,footnote_text
0,R,Revised
1,X,Data unavailable due to the 2025 lapse in appropriations


Non-numeric values:


,series_id,year,period,value,footnote_codes,footnote_text
0,CUUR0000SA0,2025,M10,-,X,Data unavailable due to the 2025 lapse in appropriations
1,CUURS12ASA0,2025,M10,-,X,Data unavailable due to the 2025 lapse in appropriations


Footnote codes used in the three series: {'': 2811, 'X': 2}


**Special values**
- Only two non-numeric values in the three series used: October 2025 = `-`, footnote `X` "Data unavailable due to the 2025 lapse in appropriations" (the federal shutdown). BLS never collected it, so it won't be revised in.
- `R` (revised) exists in the footnote list but isn't used in these series.

## 5. Coverage: the project window (2018 → latest)

In [5]:
ny_m = cpi[(cpi.series_id == "CUURS12ASA0") & (cpi.year >= 2018)]
grid = ny_m.pivot(index="year", columns="period", values="value_num")
display(grid.round(1))
months = grid[[f"M{m:02d}" for m in range(1, 13)]]
print("Missing months 2018+:", [(y, c) for y in months.index for c in months.columns if pd.isna(months.loc[y, c]) and not (y == 2026 and c > "M08")])

period,M01,M02,M03,M04,M05,M06,M07,M08,M09,M10,M11,M12,M13
year,,,,,,,,,,,,,
2018,270.8,272.2,272.2,273.0,274.0,274.2,274.1,274.4,275.5,275.1,274.5,273.8,273.6
2019,275.1,275.8,276.6,277.4,278.1,278.8,278.8,279.4,279.3,279.3,279.5,279.8,278.2
2020,282.0,282.6,282.0,280.6,282.1,282.3,283.6,283.5,284.6,284.1,283.3,284.4,282.9
2021,285.5,286.5,287.5,289.5,291.0,293.9,293.6,293.9,295.5,296.5,297.5,296.9,292.3
2022,300.2,301.2,305.0,307.8,309.2,313.6,312.6,313.3,313.9,314.3,315.0,315.7,310.1
2023,318.2,319.3,319.0,319.2,320.0,321.3,322.5,324.4,325.6,325.3,324.5,324.7,322.0
2024,328.0,328.6,329.8,331.3,332.6,334.8,335.6,336.5,337.9,338.2,338.5,338.6,334.2
2025,341.1,342.3,342.5,344.0,343.9,346.3,346.4,347.3,348.1,NaN,348.6,350.0,345.5
2026,350.9,353.4,356.2,359.8,361.4,360.5,362.2,362.3,NaN,NaN,NaN,NaN,NaN


Missing months 2018+: [(2025, 'M10')]


**Coverage findings**
- 2018 → August 2026 complete except October 2025. 2026 has no annual average yet (8 months).

## 6. Relationships: which CPI value goes with which pay figure?

OEWS wages are for May of each year (a 3-year rolling sample, nb 03); ACS wages are past 12 months in each response year, already brought to 2024 dollars by `ADJINC` (nb 10). Compare the May index with the annual average.

In [6]:
y = pd.DataFrame({"may": months["M05"], "annual_avg": grid["M13"]}).dropna()
y["may_vs_annual_%"] = (y.may / y.annual_avg - 1) * 100
display(y.round(2))

,may,annual_avg,may_vs_annual_%
year,,,
2018,274.00,273.64,0.13
2019,278.07,278.16,-0.03
2020,282.09,282.92,-0.29
2021,290.99,292.30,-0.45
2022,309.24,310.14,-0.29
2023,320.00,322.00,-0.62
2024,332.63,334.21,-0.47
2025,343.89,345.51,-0.47


**Relationship findings**
- The May index sits within 0.6% of the annual average every year 2018–2025. Either works for OEWS; the annual average is the standard (BLS and NYS DOL deflate annual wages with it) and also fits ACS's 12-month incomes.

## 7. Tie-out: do the published annual averages match the months?

BLS publishes the annual average as period `M13`. Recompute it from the 12 months. For 2025, October is missing; see what BLS did.

In [7]:
calc = months.mean(axis=1, skipna=True)
t = pd.DataFrame({"published_M13": grid["M13"], "mean_of_months": calc.round(3), "months_used": months.notna().sum(axis=1)})
t["diff"] = (t.published_M13 - t.mean_of_months).round(3)
display(t)
semi = cpi[(cpi.series_id == "CUUSS12ASA0") & (cpi.year >= 2018)].pivot(index="year", columns="period", values="value_num")
print("Semiannual series: annual average S03 vs monthly M13:"); display(pd.concat([semi["S03"], grid["M13"]], axis=1).round(3))

,published_M13,mean_of_months,months_used,diff
year,,,,
2018,273.641,273.640,12,0.001
2019,278.164,278.164,12,0.000
2020,282.920,282.920,12,0.000
2021,292.303,292.303,12,0.000
2022,310.141,310.141,12,0.000
2023,321.998,321.998,12,0.000
2024,334.209,334.208,12,0.001
2025,345.511,345.511,11,0.000
2026,NaN,358.333,8,NaN


Semiannual series: annual average S03 vs monthly M13:


,S03,M13
year,,
2018,273.641,273.641
2019,278.164,278.164
2020,282.920,282.920
2021,292.303,292.303
2022,310.141,310.141
2023,321.998,321.998
2024,334.209,334.209
2025,345.511,345.511
2026,NaN,NaN


**Tie-out findings**
- Published annual averages = mean of the months for 2018–2024 (diff ≤ 0.001, rounding) , and the half-yearly series gives exactly the same annual averages .
- **2025:** BLS's published annual average 345.511 = the mean of the 11 months it has (October left out). So the 2025 average is official and usable; footnote it.

## 8. Fitness for purpose: the deflator

To show pay in 2025 dollars: `real = nominal × CPI(2025) / CPI(year)`, annual averages, NY metro. Compare NY inflation with the U.S. average.

In [8]:
us = cpi[(cpi.series_id == "CUUR0000SA0") & (cpi.period == "M13") & (cpi.year >= 2018)].set_index("year").value_num
nyA = grid["M13"].dropna()
BASE = 2025
defl = pd.DataFrame({"ny_cpi": nyA, "to_2025_dollars": nyA[BASE] / nyA, "ny_yoy_%": nyA.pct_change() * 100,
                     "us_yoy_%": us.pct_change() * 100})
display(defl.round(3))
print(f"Prices 2019 -> 2025: NY metro +{(nyA[2025] / nyA[2019] - 1) * 100:.1f}% | U.S. +{(us[2025] / us[2019] - 1) * 100:.1f}%")
print(f"A $60,000 wage in May 2019 is ${60000 * nyA[2025] / nyA[2019]:,.0f} in 2025 dollars.")

,ny_cpi,to_2025_dollars,ny_yoy_%,us_yoy_%
year,,,,
2018,273.641,1.263,NaN,NaN
2019,278.164,1.242,1.653,1.812
2020,282.920,1.221,1.710,1.234
2021,292.303,1.182,3.316,4.698
2022,310.141,1.114,6.103,8.003
2023,321.998,1.073,3.823,4.116
2024,334.209,1.034,3.792,2.949
2025,345.511,1.000,3.382,2.631


Prices 2019 -> 2025: NY metro +24.2% | U.S. +25.9%
A $60,000 wage in May 2019 is $74,527 in 2025 dollars.


**Fitness findings**
- **Q4 / O4: fit.** Deflator to 2025 dollars: 2018 × 1.263, 2019 × 1.242, 2020 × 1.221, 2021 × 1.182, 2022 × 1.114, 2023 × 1.073, 2024 × 1.034.
- NY prices rose 24.2% from 2019 to 2025 (U.S. 25.9%); NY inflation ran below the U.S. through 2023, above it in 2024–2025. Example: a $60,000 wage in 2019 = $74,527 in 2025 dollars, so a job whose pay went from $60k to $70k actually lost buying power.
- **Limits:** metro area, not NYC; one basket for everyone (a student's costs, rent above all, may rise faster); no NYC-only or borough index exists.

## 9. Issues list and proposed staging table

### 9. Issues list

| # | Issue | Rows affected | Proposed treatment (decided in staging) |
|---|---|---|---|
| 1 | October 2025 missing (`-`, footnote X) | 1 month per series | `value = NULL`, keep `footnote_code`; no interpolation (BLS's 2025 annual average already excludes it) |
| 2 | Values are padded text | all | Strip, cast to `numeric(8,3)` |
| 3 | Annual average mixed in with months (`M13`) | 1 row per year | `period_type` = `month` / `annual_avg`; `month` = 1–12 or NULL |
| 4 | Metro area, not NYC | all | Label "New York metro area prices" |
| 5 | File overwrites itself; 2026 incomplete | latest months | Snapshot file names (already done); refresh yearly once `M13` is posted (mid-January) |

### Proposed staging table

**`stg_bls__cpi`**
- **Grain / key:** `(series_id, year, period)`
- **Columns:** `series_id`, `area_code` (`S12A` / `0000`), `area_name`, `year`, `period`, `period_type`, `month`, `index_value` numeric (1982–84 = 100), `footnote_code`
- **Filters:** series `CUURS12ASA0` (NY metro) and `CUUR0000SA0` (U.S., for comparison), years ≥ 2015 (row count below). The deflator (`to_2025_dollars`) is computed in core from the annual averages, with the base year as a parameter.

In [9]:
stg = cpi[cpi.series_id.isin(["CUURS12ASA0", "CUUR0000SA0"]) & (cpi.year >= 2015)]
print(f"stg_bls__cpi: {len(stg):,} rows ({stg.series_id.nunique()} series, {stg.year.min()}-{stg.year.max()}), "
      f"non-numeric values kept as NULL: {stg.value_num.isna().sum()}")

stg_bls__cpi: 302 rows (2 series, 2015-2026), non-numeric values kept as NULL: 2
